# Fine-tuning de BETO — Clasificador de solicitudes ciudadanas CDMX

**Proyecto:** Clasificador de solicitudes ciudadanas en español con generación automática de tickets en CDMX (UNIR, Maestría en IA).

**Sprint 2 — Épica E3 (Modelo BETO), HU-06 y HU-07.**

Este notebook realiza el fine-tuning de BETO (`dccuchile/bert-base-spanish-wwm-cased`) sobre el corpus preparado con `01_preparar_corpus.py` y lo evalúa con las **mismas particiones y métricas** que el baseline TF-IDF + regresión logística, para una comparación justa (sección 3.8 del documento).

**Requisitos:** ejecutar en Google Colab con GPU (`Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`). Subir los archivos `train.csv`, `val.csv` y `test.csv` generados por `01_preparar_corpus.py` (celda 2).


In [ ]:
# 1. Instalación de dependencias
!pip -q install transformers datasets evaluate accelerate scikit-learn
import torch
print("GPU disponible:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
GPU disponible: True


In [ ]:
# 2. Subir el corpus (train.csv, val.csv, test.csv de la carpeta corpus/)
from google.colab import files
subidos = files.upload()   # seleccionar los 3 CSV


Saving test.csv to test.csv
Saving train.csv to train.csv
Saving val.csv to val.csv


In [ ]:
# 3. Carga de datos
import pandas as pd
tr = pd.read_csv("train.csv"); va = pd.read_csv("val.csv"); te = pd.read_csv("test.csv")
etiquetas = sorted(tr["categoria"].unique())
id2label = dict(enumerate(etiquetas)); label2id = {v: k for k, v in id2label.items()}
for d in (tr, va, te): d["label"] = d["categoria"].map(label2id)
print(f"train={len(tr)} val={len(va)} test={len(te)}")
print(tr["categoria"].value_counts())


train=1012 val=217 test=217
categoria
alumbrado_publico     291
baches_y_pavimento    279
fuga_de_agua          204
seguridad             189
recoleccion_basura     49
Name: count, dtype: int64


In [ ]:
# 4. Tokenización
from transformers import AutoTokenizer
from datasets import Dataset
MODELO = "dccuchile/bert-base-spanish-wwm-cased"   # BETO (Cañete et al., 2020)
tok = AutoTokenizer.from_pretrained(MODELO)
def tokenizar(b): return tok(b["texto"], truncation=True, max_length=128)
ds = {n: Dataset.from_pandas(d[["texto","label"]]).map(tokenizar, batched=True)
      for n, d in {"train": tr, "val": va, "test": te}.items()}


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Map:   0%|          | 0/1012 [00:00<?, ? examples/s]

Map:   0%|          | 0/217 [00:00<?, ? examples/s]

Map:   0%|          | 0/217 [00:00<?, ? examples/s]

In [ ]:
# 5. Fine-tuning con ponderación de clases (desbalance, sección 3.3.6)
import numpy as np, torch
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score

pesos = torch.tensor(
    (len(tr) / tr["categoria"].value_counts().reindex(etiquetas)).values,
    dtype=torch.float)

class TrainerPonderado(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            out.logits, labels, weight=pesos.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def metricas(p):
    pred = np.argmax(p.predictions, axis=1)
    return {"f1_macro": f1_score(p.label_ids, pred, average="macro")}

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO, num_labels=len(etiquetas), id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir="beto_cdmx", eval_strategy="epoch", save_strategy="epoch",
    learning_rate=3e-5, per_device_train_batch_size=16,
    per_device_eval_batch_size=32, num_train_epochs=6, weight_decay=0.01,
    load_best_model_at_end=True, metric_for_best_model="f1_macro",
    logging_steps=10, seed=42, report_to="none")

trainer = TrainerPonderado(model=modelo, args=args,
    train_dataset=ds["train"], eval_dataset=ds["val"],
    processing_class=tok, data_collator=DataCollatorWithPadding(tok),
    compute_metrics=metricas)
trainer.train()


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

Epoch,Training Loss,Validation Loss,F1 Macro
1,0.505002,0.431268,0.897864
2,0.275271,0.417548,0.900238
3,0.090922,0.301189,0.910891
4,0.027817,0.299980,0.927317
5,0.009456,0.343021,0.912570
6,0.004621,0.356976,0.916377


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=384, training_loss=0.2762447672430426, metrics={'train_runtime': 214.4268, 'train_samples_per_second': 28.317, 'train_steps_per_second': 1.791, 'total_flos': 213335367665616.0, 'train_loss': 0.2762447672430426, 'epoch': 6.0})

In [ ]:
# 6. Calibración por temperatura con validación (sección 3.3.8)
import torch.nn.functional as F
logits_va = torch.tensor(trainer.predict(ds["val"]).predictions)
y_va = torch.tensor(va["label"].values)
T = torch.nn.Parameter(torch.ones(1))
opt = torch.optim.LBFGS([T], lr=0.05, max_iter=100)
def paso():
    opt.zero_grad()
    loss = F.cross_entropy(logits_va / T, y_va)
    loss.backward(); return loss
opt.step(paso)
print(f"Temperatura óptima: {T.item():.3f}")


Temperatura óptima: 0.363


In [ ]:
# 7. Evaluación final sobre prueba: mismas métricas que el baseline
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

UMBRAL_ABSTENCION = 0.60
logits_te = torch.tensor(trainer.predict(ds["test"]).predictions)
proba = F.softmax(logits_te / T.detach(), dim=1).numpy()
pred = np.array([id2label[i] for i in proba.argmax(1)])
conf = proba.max(1)
abstiene = conf < UMBRAL_ABSTENCION
y_true = te["categoria"].values

def ece(y_true, y_pred, confs, n_bins=10):
    ok = (y_true == y_pred).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); err = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (confs > lo) & (confs <= hi)
        if m.sum(): err += m.sum()/len(confs) * abs(ok[m].mean() - confs[m].mean())
    return err

print(f"Accuracy:            {accuracy_score(y_true, pred):.3f}")
print(f"F1 macro:            {f1_score(y_true, pred, average='macro'):.3f}  (criterio: >= 0.80)")
print(f"ECE:                 {ece(y_true, pred, conf):.3f}")
print(f"Cobertura automática: {1 - abstiene.mean():.1%}")
m = ~abstiene
if m.sum():
    print(f"F1 macro emitidos:   {f1_score(y_true[m], pred[m], average='macro'):.3f}")
print()
print(classification_report(y_true, pred, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_true, pred, labels=etiquetas),
                   index=etiquetas, columns=etiquetas))


Accuracy:            0.958
F1 macro:            0.916  (criterio: >= 0.80)
ECE:                 0.032
Cobertura automática: 97.9%
F1 macro emitidos:   0.988

                    precision    recall  f1-score   support

 alumbrado_publico      1.000     1.000     1.000         4
baches_y_pavimento      1.000     1.000     1.000         9
      fuga_de_agua      0.947     0.947     0.947        19
recoleccion_basura      1.000     0.500     0.667         2
         seguridad      0.933     1.000     0.966        14

          accuracy                          0.958        48
         macro avg      0.976     0.889     0.916        48
      weighted avg      0.960     0.958     0.955        48

                    alumbrado_publico  baches_y_pavimento  fuga_de_agua  \
alumbrado_publico                   4                   0             0   
baches_y_pavimento                  0                   9             0   
fuga_de_agua                        0                   0            18   

In [ ]:
# 8. Guardar predicciones y modelo para la comparación formal (HU-07)
sal = te[["id_registro","texto","categoria"]].copy()
sal["prediccion"] = pred; sal["confianza"] = conf.round(3)
sal["requiere_revision"] = abstiene
sal.to_csv("predicciones_test_beto.csv", index=False)
trainer.save_model("beto_cdmx_final"); tok.save_pretrained("beto_cdmx_final")
!zip -qr beto_cdmx_final.zip beto_cdmx_final
from google.colab import files
files.download("predicciones_test_beto.csv")
print("Descarga predicciones_test_beto.csv y colócalo en la carpeta resultados/ del proyecto.")
print("Descarga también beto_cdmx_final.zip si quieres conservar el modelo.")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga predicciones_test_beto.csv y colócalo en la carpeta resultados/ del proyecto.
Descarga también beto_cdmx_final.zip si quieres conservar el modelo.


## Comparación con el baseline (HU-07)

Con `predicciones_test_beto.csv` en la carpeta `resultados/` del proyecto, ejecuta en tu máquina `03_comparar_modelos.py` para obtener la tabla comparativa baseline vs. BETO (F1 macro, F1 por clase, ECE, cobertura) y aplicar el **criterio de decisión** de la sección 4.6: si BETO no supera de manera consistente al baseline, se conserva TF-IDF + regresión logística por su menor costo.

**Advertencia metodológica:** con el corpus actual (~190 ejemplos útiles tras deduplicar plantillas), los resultados de BETO tendrán alta varianza; la comparación será orientativa hasta ampliar el corpus (ver plan, Fase 1).
